In [2]:
!pip install lxml
!pip install DrissionPage
!pip install curl_cffi
from curl_cffi import requests
from bs4 import BeautifulSoup
import time
from DrissionPage import ChromiumPage, ChromiumOptions
import pandas as pd
import os
import random

In [3]:
def save_to_csv(data_list, filename):


    df = pd.DataFrame(data_list)
    
    # Kiểm tra xem file đã tồn tại trên máy chưa
    file_exists = os.path.isfile(filename)
    
    # Lưu ra CSV
    # Nếu file chưa tồn tại (file_exists = False), header=True sẽ in ra tên cột.
    # Nếu file đã tồn tại, header=False để không in lại tên cột ở giữa file.
    df.to_csv(filename, mode='a', index=False, header=not file_exists, encoding='utf-8')
    
    print(f"[CHECKPOINT] Đã lưu thành công {len(df)} dòng vào '{filename}'.")
    
    # giải phóng RAM
    data_list.clear()

In [4]:
def get_link_books(start=1,end=1,url="url"):
    book_links = []
    
    for i in range(start,end+1):
        print(f"Đang cào trang số {i} ...")
        current_url = url + str(i)
        # headers = {
        #     'User-Agent': random.choice(USER_AGENTS_LIST)
        # }  
        try:
            response = requests.get(current_url, 
                                    impersonate="chrome124", # Lệnh ma thuật giả lập 100% trình duyệt Chrome 124
                                    timeout=15)
            if response.status_code != 200:
                print(f"Lỗi truy cập trang : {response.status_code}")
                continue     
                
            soup = BeautifulSoup(response.text, 'lxml')
            book_tags = soup.find_all('a', {"class":"bookTitle"})
            for link in book_tags:
                href = link.get('href')
                if href:
                    full_link = "https://www.goodreads.com"+href
                    book_links.append(full_link)   
            time.sleep(random.uniform(1, 3))     
        except Exception as e:
            print(e)
            continue
            
    return book_links

In [5]:
def get_information_in_book(book_links,file_name):
    books_data = []
    
    i = 0
    for count,url in enumerate(book_links):
        # headers = {
        #     'User-Agent': random.choice(USER_AGENTS_LIST)
        # } 
        
        i+=1
        print(f"Đang cào thông tin sách số {i}: {url}")
        try:
            response = requests.get(url, 
                                    impersonate="chrome124", # Lệnh ma thuật giả lập 100% trình duyệt Chrome 124
                                    timeout=15)
            if response.status_code != 200:
                print(f"Lỗi {response.status_code} tại link: {url}")
                continue 

            html_content = response.text

            soup = BeautifulSoup(html_content, "lxml")
            
            book_id_tag = url.split('/')[-1]
            book_id = book_id_tag.split('-')[0] if book_id_tag.split('-')[0].isdigit() else book_id_tag.split('.')[0]

           
            chuoi = "kca://work/amzn1.gr.work.v1."
            if chuoi in html_content:
                work_id_tag = html_content.split(chuoi)[1].split('"')[0].split("'")[0] 
                work_id = chuoi + work_id_tag 
            else:
                work_id = None 
            
            title_tag = soup.find("h1", {"data-testid": "bookTitle"})
            title = title_tag.text.strip() if title_tag else "N/A"
            
            author_tag = soup.find("span", {"data-testid": "name"})
            author = author_tag.text.strip() if author_tag else "N/A"

            rating_tag = soup.find("div", {"class": "RatingStatistics__rating"})
            avg_rating = rating_tag.text.strip() if rating_tag else 0

            desc_tag = soup.find("span", {"class": "Formatted"})
            description = desc_tag.text.strip() if desc_tag else "None"

            genre_buttons = soup.find_all("span", {"class": "BookPageMetadataSection__genreButton"})
            genres_list = [a.text.strip() for a in genre_buttons]
            genres = ",".join(genres_list)

            page_count_tag = soup.find("p",{"data-testid":"pagesFormat"})
            if page_count_tag:                
                text_raw = page_count_tag.text                                
                chuoi_so = "".join([c for c in text_raw if c.isdigit()])
                page_count = int(chuoi_so) if chuoi_so else 0
            else:                
                page_count = 0
            
            publish_date_tag = soup.find("p",{"data-testid":"publicationInfo"})
            publish_date = publish_date_tag.text.strip().split("published")[-1].strip() if publish_date_tag else "None"

            image_url_tag = soup.find("img",{"class":"ResponsiveImage"})
            image_url = image_url_tag.get('src') if image_url_tag else "None"

            total_ratings_tag = soup.find("span",{"data-testid":"ratingsCount"})
            total_ratings = total_ratings_tag.text.strip().split("\xa0")[0] if total_ratings_tag else "0"

            total_reviews_tag = soup.find("span",{"data-testid":"reviewsCount"})
            total_reviews = total_reviews_tag.text.strip().split("\xa0")[0] if total_reviews_tag else "0"

            book_id = int(book_id)
            
            total_ratings = total_ratings.replace(",", "").replace(".", "").strip()
            total_ratings = int(total_ratings)
            
            total_reviews = total_reviews.replace(",", "").replace(".", "").strip()
            total_reviews = int(total_reviews)
            
            avg_rating = float(avg_rating)
            
            
            books_data.append({
                "book_id": book_id,
                "work_id":work_id,
                "title": title,
                "author": author,
                "total_ratings":total_ratings,
                "total_reviews":total_reviews,
                "avg_rating": avg_rating,
                "description": description,
                "genres": genres,
                "page_count":page_count,
                "publish_date":publish_date,
                "image_url":image_url
            })
            # cào được 10 cuốn thì lưu luôn
            if (count + 1) % 10 == 0 and count != 0:
                save_to_csv(books_data, file_name)
            time.sleep(random.uniform(3, 7))
        except Exception as e:
            print(f"Có lỗi khi cào {url}: {e}")
            continue 
    if books_data:
        save_to_csv(books_data, file_name)

In [6]:
def get_api_key(book_id):
    url = f"https://www.goodreads.com/book/show/{book_id}"
    token = None
    
    try:
        co = ChromiumOptions().set_local_port(9222)
        page = ChromiumPage(co)
        
        page.listen.start('graphql') 
        page.get(url)
        page.wait(3) 
        
        close_popup = page.ele('@aria-label=Close', timeout=2) 
        if close_popup:
            close_popup.click(by_js=True)
            page.wait(2)
        
        more_button = None
        for i in range(3): 
            more_button = page.ele('@aria-label=Tap to show more reviews and ratings', timeout=1)
            if more_button:
                break
            page.scroll.down(600)
            page.wait(0.5) 
            
        if more_button:    
            more_button.scroll.to_see(center=True)
            page.listen.clear() 
            more_button.click(by_js=True) 

            for step in range(10): 
                packet = page.listen.wait(timeout=2)          
                if packet:
                    if packet.method.upper() == 'OPTIONS':            
                        continue
        
                    api_key = packet.request.headers.get('x-api-key')
                    auth_header = packet.request.headers.get('authorization') or packet.request.headers.get('Authorization')
                    
                    token = api_key or auth_header
                    
                    if token:
                        print(f"Đã lấy được Token loại: {'x-api-key' if api_key else 'Authorization'}")
                        break
        else:
            print(f"Sách {book_id}: Không tìm thấy nút More reviews.")

        page.listen.stop()
        page.quit()
        return token
        
    except Exception as e:
        print(f"Lỗi tại sách {book_id}: {e}")
        page.listen.stop()
        return None



In [7]:
def get_reviews_graphql(books_list,file_name ,max_pages=10):
    current_token = "api_key"
    for i,book in enumerate(books_list):
        user_reviews = []
        
        headers = {
            'accept': '*/*',
            'accept-language': 'en-US,en;q=0.9',
            'x-api-key': current_token, 
            'content-type': 'application/json',
            'origin': 'https://www.goodreads.com',
            #'user-agent': random.choice(USER_AGENTS_LIST)
        }
        
        work_id = book.get('work_id')       
        book_id = book.get('book_id')
        
        if not work_id:
            print(f"Bỏ qua sách {book.get('title')} vì không có mã work_id.")
            continue
            
        print(f"\nĐang cào review sách {i+1} : {book.get('title')}")

        json_data = {
            'operationName': 'getReviews',
            'variables': {
                'filters': {
                    'resourceType': 'WORK',
                    'resourceId': work_id, 
                },
                'pagination': {
                    'limit': 30, 
                },
            },
        'query': 'query getReviews($filters: BookReviewsFilterInput!, $pagination: PaginationInput) {\n  getReviews(filters: $filters, pagination: $pagination) {\n    edges {\n      node {\n        creator {\n          id: legacyId\n          name\n        }\n        rating\n        text\n      }\n    }\n    pageInfo {\n      nextPageToken\n    }\n  }\n}'
        }

 
        curent_page = 0
        while curent_page < max_pages:
            print(f" --> Đang cào review trang {curent_page + 1}/{max_pages}...")  
            
            response = requests.post(
                'https://kxbwmqov6jgg3daaamb744ycu4.appsync-api.us-east-1.amazonaws.com/graphql',
                headers=headers,
                impersonate="chrome124",
                json=json_data,
                timeout=15
            )
            
            if response.status_code == 401:
                print("Token hết hạn ...")

                new_api_key = get_api_key(book_id) 
                
                if new_api_key:
                    current_token = new_api_key
                    headers['x-api-key'] = new_api_key
                    print("Đã lấy được Token mới ...")
                    continue
                else:
                    print("Không thể lấy key mới. Nhảy sang sách khác.")
                    break
                    
            elif response.status_code != 200:
                print(f"Lỗi: {response.status_code}")
                break
                
            data = response.json()
            
            try:
                reviews_list = data['data']['getReviews']['edges']          
                for node in reviews_list:
                    thong_tin = node['node']
                    user = thong_tin.get('creator', {})            
                    
                    user_id = user.get('id', 'N/A')
                    if user_id != 'N/A':
                        user_reviews.append({
                            "book_id": book_id,
                            "user_id": user_id,
                            "user_name": user.get('name', 'Anonymous'),
                            "rating": thong_tin.get('rating', 0),
                            "comment": thong_tin.get('text', '')
                        })
 

                next_cursor = data['data']['getReviews']['pageInfo'].get('nextPageToken')
                if next_cursor:
                    json_data['variables']['pagination']['after'] = next_cursor
                    curent_page += 1 
                else:
                    print(" -> Đã vét sạch mọi review của cuốn sách này!")
                    break
                    
            except (KeyError,TypeError):
                print(" -> Không tìm thấy dữ liệu review trong JSON.")
                break
                
            time.sleep(1) 
        print(f"đã cào được {len(user_reviews)} dòng")
        save_to_csv(user_reviews,file_name)
        time.sleep(random.uniform(1, 3))


In [8]:
# books_data = {
#         "book_id": 2767052,
#         "work_id":work_id,
#         "title": "The Hunger Games",
#         "author": "Suzanne Collins",
#         "image_url": "https://images-na.ssl-images-amazon.com/...", 
#         "page_count": 374,       
#         "publish_year": 2008,    
#         "avg_rating": 4.33,
#         "total_ratings":10000,
#         "total_reviews":11111,
#         "description": "Winning means fame and fortune...",        
#        "genres": "Young Adult, Dystopia, Fiction",         
# }

# user_data = [
#     {"user_id": "U_123", "book_id": "2767052", "star": 5, "comment": "Quá hay!"},
#     {"user_id": "U_123", "book_id": "3", "star": 4, "comment": "Cũng được"},
#     {"user_id": "U_456", "book_id": "2767052", "star": 1, "comment": "Tệ!"}
# ]

# crawl :
# lấy link các sách từng trang
# vào từng link sách lấy các thông tin
# lấy comment của các user của từng sách
# quay lại bước đầu và sang trang khác


In [ ]:
# final 
url = "https://www.goodreads.com/list/show/1.Best_Books_Ever?page="
page_start = 11
page_end = 11
max_pages = 30

for page in range(page_start,page_end+1):
    
    book_file_name = os.path.join("Data",f"books_data_{page}.csv")
    reviewer_file_name = os.path.join("Data",f"reviewers_data_{page}.csv")

    
    # 1.Lấy danh sách link
    books_link = get_link_books(page, page, url)
    if not books_link :
        continue
    
    # 2.Cào thông tin sách và lưu dần vào file 'books_data_{page}.csv'
    get_information_in_book(books_link,book_file_name) 
    
    # 3.Đọc file CSV lấy toàn bộ danh sách sách đã cào được đi lấy Review lưu vào 'reviewers_data_{page}.csv'
    try :
        df = pd.read_csv(book_file_name)
        df = df.where(pd.notna(df), None)
        df_sach = df.to_dict('records')         
        get_reviews_graphql(df_sach,reviewer_file_name,max_pages) # mặc định cào 10 trang x 30 comments
    except Exception as e:
        print(f"Lỗi phần Review trang {page}: {e}")
        
    time.sleep(5)
    
print(" TẤT CẢ ĐÃ HOÀN TẤT! ")